In [1]:
from google.colab import drive
import os

# 1. 구글 드라이브 마운트 (팝업창에서 '허용'을 눌러주세요)
drive.mount('/content/drive')

# 2. 경로 설정
zip_path = '/content/drive/MyDrive/fine_tuning_dataset_v3.1.zip'
target_dir = '/content'  # 코랩 최상위에 풀면 알아서 v3.1 폴더가 생성됩니다.

print(f"\n구글 드라이브의 '{os.path.basename(zip_path)}' 파일을 코랩 로컬로 푸는 중입니다...")
print("잠시만 기다려주세요! \n")

if os.path.exists(zip_path):
    # 3. 초고속 압축 해제 (-q: 조용히, -o: 덮어쓰기, -d: 지정 경로에 풀기)
    !unzip -q -o "{zip_path}" -d "{target_dir}"

    # 4. 압축이 잘 풀렸는지 최종 확인
    check_path = '/content/fine_tuning_dataset_v3.1'
    if os.path.exists(check_path):
        print("-" * 50)
        print("압축 해제 완료! 데이터셋이 코랩 로컬에 준비되었습니다.")
        print(f"데이터 위치: {check_path}")
        print("왼쪽 파일 탐색기를 새로고침해서 'images'와 'labels'가 잘 들어왔는지 확인해 보세요!")
    else:
        print("\n앗, 압축은 풀렸으나 예상된 폴더를 찾지 못했습니다. 파일 탐색기를 새로고침해서 폴더 이름을 확인해 보세요.")
else:
    print("\n오류: 드라이브에서 zip 파일을 찾을 수 없습니다.")
    print("구글 드라이브 최상위 경로에 'fine_tuning_dataset_v3.1.zip'이 있는지 확인해 주세요.")

Mounted at /content/drive

⏳ 구글 드라이브의 'fine_tuning_dataset_v3.1.zip' 파일을 코랩 로컬로 푸는 중입니다...
데이터가 많아 1~3분 정도 소요될 수 있습니다. 잠시만 기다려주세요! ☕

--------------------------------------------------
🎉 압축 해제 완료! 데이터셋이 코랩 로컬에 완벽하게 준비되었습니다.
📁 데이터 위치: /content/fine_tuning_dataset_v3.1
왼쪽 파일 탐색기(📁)를 새로고침(🔄)해서 'images'와 'labels'가 잘 들어왔는지 확인해 보세요!


In [2]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.9 MB/s eta 0:00:00


In [3]:
from ultralytics import YOLO

model = YOLO('best.pt')

print("🚀 v3.1 오답 노트 + 1024 고해상도(CCTV용) 파인튜닝을 시작합니다...\n")

# 2. 파인튜닝 시작
results = model.train(
    data='/content/fine_tuning_dataset_v3.1/data.yaml',
    optimizer='MuSGD',
    epochs=30,
    patience=10,

    # CCTV 소형 객체 탐지용 해상도
    imgsz=1024,

    # VRAM 계산 결과에 따른 최적화 배치 사이즈
    batch=128,
    workers=8,

    # 기존 지식을 보존하기 위한 학습률 설정
    lr0=0.005,
    lrf=0.01,
    warmup_epochs=1.0,

    momentum=0.937,
    weight_decay=0.0005,
    freeze=0,

    # 모자/안경 오인식 강력 페널티
    cls=3.0,

    save_period=5,
    mosaic=1.0,
    mixup=0.05,

    project='runs/detect',
    name='helmet_1024_v3.1_run'
)

print("\n🎉 파인튜닝 완료! 'runs/detect/helmet_1024_v3.1_run'에서 결과를 확인하세요.")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
🚀 v3.1 오답 노트 + 1024 고해상도(CCTV용) 파인튜닝을 시작합니다...

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=128, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=3.0, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/fine_tuning_dataset_v3.1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript,

In [8]:
from ultralytics import YOLO
import cv2

model = YOLO('/content/runs/detect/runs/detect/data_v3.1/weights/best.pt')

# 영상 경로 설정
video_path = '/content/cctv.mp4'

# 추론 실행 및 결과 저장
# conf=0.3: 신뢰도 임계값, 원하시는 수치로 조정 가능합니다.
# save=True: 결과를 영상으로 저장합니다.
results = model.predict(
    source=video_path,
    conf=0.3,
    save=True,
    imgsz=1024,
    device=0,           # GPU 사용
    stream=True         # 대용량 영상 처리를 위한 스트리밍 방식
)

# 결과 출력
# stream=True를 사용하면 generator 객체가 반환되므로 루프를 돌려야 합니다.
for r in results:
    # 각 프레임별로 추론 결과를 처리하거나 시각화할 수 있습니다.
    # r.plot()을 사용하면 바운딩 박스가 그려진 이미지가 생성됩니다.
    annotated_frame = r.plot()

    # 화면 표시를 원하시면 cv2.imshow를 쓰지만, 코랩에서는
    # 자동으로 save=True 설정에 의해 파일로 저장됩니다.
    pass

print("영상 추론 및 결과 저장 완료!")


video 1/1 (frame 1/102) /content/cctv.mp4: 576x1024 (no detections), 14.0ms
video 1/1 (frame 2/102) /content/cctv.mp4: 576x1024 (no detections), 11.8ms
video 1/1 (frame 3/102) /content/cctv.mp4: 576x1024 (no detections), 11.8ms
video 1/1 (frame 4/102) /content/cctv.mp4: 576x1024 (no detections), 11.7ms
video 1/1 (frame 5/102) /content/cctv.mp4: 576x1024 (no detections), 11.5ms
video 1/1 (frame 6/102) /content/cctv.mp4: 576x1024 (no detections), 11.6ms
video 1/1 (frame 7/102) /content/cctv.mp4: 576x1024 (no detections), 11.6ms
video 1/1 (frame 8/102) /content/cctv.mp4: 576x1024 (no detections), 11.6ms
video 1/1 (frame 9/102) /content/cctv.mp4: 576x1024 (no detections), 11.6ms
video 1/1 (frame 10/102) /content/cctv.mp4: 576x1024 (no detections), 11.5ms
video 1/1 (frame 11/102) /content/cctv.mp4: 576x1024 (no detections), 11.6ms
video 1/1 (frame 12/102) /content/cctv.mp4: 576x1024 (no detections), 11.5ms
video 1/1 (frame 13/102) /content/cctv.mp4: 576x1024 (no detections), 11.4ms
video 1